# Temporary: Audit Review

Review and correct the pre-filled codes and notes of the four audits before they are reported. This notebook only reads
and writes the sample CSVs that section 4 of `2_b_comment_matching_analysis.ipynb` uses, plus its own progress file.
**Delete this notebook and `_temp_audit_review_progress.csv` once every row is reviewed.**

Loop: run a `review(...)` cell. It shows the next unreviewed row (uncoded rows first, then `[recheck]`/`[unsure]` rows) with the screenshot, the current
codes and a `confirm(...)` line. Paste that line into the cell below, edit the codes or note if needed, run it, then run
the review cell again. `confirm` overwrites the row and marks it reviewed; the printed note already drops the flag
prefix. `review(section, model, index=n)` revisits a row by `sample_index`; `overview(...)` lists a section as a table.

In [ ]:
import sys
from pathlib import Path

import pandas as pd
import yaml
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "utils" / "paths.py").exists())
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))
from utils.paths import RESULTS_DIR
from utils.screen_display import show_screen

pd.set_option("display.max_colwidth", None)
NOTEBOOK_DIR = ROOT / "notebooks" / "3_analysis"
QUALITATIVE_DIR = RESULTS_DIR / "critiques" / "analysis" / "tables" / "qualitative_examples"
MATCHING_DIR = RESULTS_DIR / "critiques" / "matching"
PROGRESS_PATH = NOTEBOOK_DIR / "_temp_audit_review_progress.csv"  # delete together with this notebook
RUN_LABEL = "zero_shot-all_tasks"
MODEL_ORDER = ["claude4_7", "gpt5", "gemini3_1pro"]
SECTIONS = ["matched", "unmatched", "unsupported", "contradiction"]
CODEBOOK = yaml.safe_load((NOTEBOOK_DIR / "qualitative_codebook.yaml").read_text())
FIELDS = {section: list(CODEBOOK[section]["fields"]) for section in SECTIONS}
FLAGS = ("[unsure] ", "[recheck] ")  # [recheck]: the coder thinks the code itself may be wrong

# Judge relations and every LLM comment, for listing all of a model's comments on a screen (unmatched review).
RELATION_ORDER = ["equivalent", "B_broader", "A_broader", "partial_overlap", "contradiction", "related_but_different", "no_match"]
RELATION_NAMES = {"B_broader": "llm_broader", "A_broader": "human_broader"}
pair_keys = ["model_short", "expert_comment_id", "llm_comment_id"]
pairs_df, llm_df = [], []
for model in MODEL_ORDER:
    run_dir = MATCHING_DIR / model / RUN_LABEL
    labels = pd.read_parquet(run_dir / "llmjudge_matching" / f"llmjudge_matching_pairs-{RUN_LABEL}-{model}.parquet")
    comments = pd.read_parquet(run_dir / f"all_comment_pairs-{RUN_LABEL}-{model}.parquet")
    pairs_df.append(labels.assign(model_short=model)[[*pair_keys, "relation", "confidence"]])
    llm_df.append(comments.assign(model_short=model).drop_duplicates("llm_comment_id")[
        ["model_short", "screen_id", "screen_task_id", "llm_comment_id", "llm_observed_issue"]
    ])
pairs_df = pd.concat(pairs_df, ignore_index=True)
llm_df = pd.concat(llm_df, ignore_index=True)


def load(section):
    code_cols = [*FIELDS[section], "exemplar", "note"]
    sample_df = pd.read_csv(QUALITATIVE_DIR / f"{section}_review_sample.csv")
    for col in code_cols:
        if col not in sample_df:
            sample_df[col] = pd.NA
    return sample_df.astype({col: "object" for col in code_cols})


def reviewed_rows(section):
    if not PROGRESS_PATH.exists():
        return set()
    progress_df = pd.read_csv(PROGRESS_PATH)
    return set(progress_df.loc[progress_df["section"].eq(section), "row"])


def is_unsure(sample_df):
    return sample_df["note"].fillna("").astype(str).str.startswith(FLAGS)


def status():
    """Reviewed and remaining rows per section and model; `unsure_left` are flagged rows not yet reviewed."""
    rows = []
    for section in SECTIONS:
        sample_df = load(section)
        reviewed = sample_df.index.isin(list(reviewed_rows(section)))
        for model in MODEL_ORDER:
            in_model = sample_df["model_short"].eq(model)
            rows.append({
                "section": section, "model": model,
                "reviewed": int((in_model & reviewed).sum()),
                "remaining": int((in_model & ~reviewed).sum()),
                "unsure_left": int((in_model & ~reviewed & is_unsure(sample_df)).sum()),
                "uncoded": int((in_model & sample_df[FIELDS[section][0]].isna()).sum()),
            })
    return pd.DataFrame(rows)


def overview(section, model=None, unsure_only=False):
    """All codes and notes of a section as one table, for scanning before or after the row-by-row review."""
    sample_df = load(section)
    sample_df = sample_df.assign(reviewed=sample_df.index.isin(list(reviewed_rows(section))))
    if model:
        sample_df = sample_df[sample_df["model_short"].eq(model)]
    if unsure_only:
        sample_df = sample_df[is_unsure(sample_df)]
    return sample_df[["model_short", "sample_index", *FIELDS[section], "exemplar", "reviewed", "note"]]


def print_screen_comments(row):
    comments_df = llm_df[llm_df["model_short"].eq(row["model_short"]) & llm_df["screen_id"].eq(row["screen_id"])].merge(
        pairs_df[pairs_df["model_short"].eq(row["model_short"]) & pairs_df["expert_comment_id"].eq(row["expert_comment_id"])],
        on=["model_short", "llm_comment_id"], how="left",
    )
    comments_df = comments_df.assign(
        other_task=comments_df["screen_task_id"].ne(row["screen_task_id"]),
        rank=comments_df["relation"].map({relation: i for i, relation in enumerate(RELATION_ORDER)}),
    ).sort_values(["other_task", "screen_task_id", "rank", "confidence"], ascending=[True, True, True, False])
    print(f"\nAll {row['model_short']} comments on this screen:")
    for i, comment in enumerate(comments_df.itertuples(), start=1):
        relation = RELATION_NAMES.get(comment.relation, comment.relation)
        tag = f"{comment.screen_task_id}, not judged" if comment.other_task else f"{relation}, {comment.confidence:.2f}"
        print(f"  {i}. [{tag}] {comment.llm_observed_issue}")


def review(section, model, index=None, unsure_first=True):
    """Show one row with its current codes; by default the next unreviewed row (uncoded, then [unsure] rows first)."""
    global current
    sample_df = load(section)
    model_df = sample_df[sample_df["model_short"].eq(model)].sort_values("sample_index")
    if index is None:
        pending_df = model_df[~model_df.index.isin(list(reviewed_rows(section)))]
        if pending_df.empty:
            print(f"All {len(model_df)} {section} rows of {model} are reviewed.")
            return
        uncoded = pending_df[FIELDS[section][0]].isna()
        if uncoded.any():
            pending_df = pending_df[uncoded]
        elif unsure_first and is_unsure(pending_df).any():
            pending_df = pending_df[is_unsure(pending_df)]
        row = pending_df.iloc[0]
    else:
        row = model_df[model_df["sample_index"].eq(index)].iloc[0]
    current = (section, row.name)

    done = row.name in reviewed_rows(section)
    print(f"{section} | {model} #{row['sample_index']} ({'reviewed' if done else 'not reviewed'}) | "
          f"{row['app_category']} | screen {row['screen_id']}")
    print("Task:", row["task"])
    if section != "unsupported":
        print("\nExpert comment:", row["expert_observed_issue"])
    if section == "unmatched":
        print(f"Covered by {row['other_models_covered']} other model(s); judge case: {row['coverage_case']}")
        print_screen_comments(row)
    else:
        for label, col in [("LLM standard", "llm_expected_standard"), ("LLM comment", "llm_observed_issue"),
                           ("LLM fix", "llm_suggested_fix")]:
            if col in row and not pd.isna(row[col]):
                print(f"{label}:", row[col])

    note = "" if pd.isna(row["note"]) else str(row["note"])
    print("\nCurrent codes:")
    for field in [*FIELDS[section], "exemplar"]:
        print(f"  {field}: {row[field]}")
    print("Current note:", note or "(none)")
    codes = ", ".join(f"{field}={None if pd.isna(row[field]) else row[field]!r}" for field in FIELDS[section])
    exemplar = False if pd.isna(row["exemplar"]) else bool(row["exemplar"])
    print(f"\nconfirm({codes}, exemplar={exemplar}, note={next((note.removeprefix(flag) for flag in FLAGS if note.startswith(flag)), note)!r})")

    bbox = None if section == "unsupported" else row.get("expert_bounding_box")
    has_bbox = bbox is not None and not pd.isna(bbox)
    show_screen(screen_id=row["screen_id"], bboxes=[bbox] if has_bbox else None,
                labels=["Expert comment"] if has_bbox else None, colors=["tab:red"], width=380)


def confirm(note, exemplar=False, **codes):
    """Save the codes and note of the row last shown (overwriting the previous ones) and mark it reviewed."""
    section, position = current
    allowed = CODEBOOK[section]["fields"]
    if set(codes) != set(allowed):
        raise ValueError(f"{section} takes exactly these codes: {list(allowed)}")
    for field, value in codes.items():
        if str(value) not in map(str, allowed[field]):
            raise ValueError(f"{field}={value!r} is not one of {list(allowed[field])}")
    sample_df = load(section)
    sample_df.loc[position, [*codes, "exemplar", "note"]] = [*codes.values(), exemplar, note]
    sample_df.to_csv(QUALITATIVE_DIR / f"{section}_review_sample.csv", index=False)

    progress_df = pd.read_csv(PROGRESS_PATH) if PROGRESS_PATH.exists() else pd.DataFrame(columns=["section", "row"])
    progress_df = pd.concat([progress_df, pd.DataFrame([{"section": section, "row": position}])]).drop_duplicates()
    progress_df.to_csv(PROGRESS_PATH, index=False)
    model_rows = set(sample_df.index[sample_df["model_short"].eq(sample_df.loc[position, "model_short"])])
    print(f"Saved. {section} {sample_df.loc[position, 'model_short']}: "
          f"{len(reviewed_rows(section) & model_rows)}/{len(model_rows)} reviewed")


status()

In [ ]:
overview("unmatched", unsure_only=True)

## 4.1 Matched (`equivalent` pairs)

*Is the match real, and how specific and sound is the LLM's fix on its own?*

**match**
- `valid`: Same problem on the same element.
- `partial`: Overlapping but not the same - one comment is broader or covers only part of the other.
- `invalid`: Different problem or different element; the label reflects surface similarity.

**llm_fix**
- `generic`: States a direction only ("increase contrast", "make it more visible").
- `concrete`: Names the element and the specific change ("darken the section header text", "add a chevron to each row").
- `specified`: Concrete plus a measurable target or ready-to-use content (a contrast ratio, a size in pt/dp, a named standard level such as WCAG AA, exact replacement copy - also when given as "e.g.").

**fix_sound**
- `sound`: Would fix the problem without obvious side effects; alternatives are fine if each would fix it.
- `partly`: Treats only part of the problem, or at least one listed alternative would not fix it.
- `unsound`: Would not fix the problem, introduces a new one, or relies on content or behaviour not on the screen.

**issue_type**
- `contrast_legibility`: Text or icons hard to read - colour contrast, small or light text.
- `hierarchy_emphasis`: Wrong visual weight - an important element is not prominent, or an unimportant one dominates.
- `layout_spacing`: Alignment, margins, whitespace, balance, clipped or cut-off content.
- `affordance`: Unclear whether or how an element is interactive, or what kind of control it is.
- `touch_target`: Controls too small or too close together to tap reliably.
- `navigation`: Missing or unclear way to move between screens (back, close, menu structure).
- `labeling_content`: Wording, labels, terminology, icon meaning, copy errors.
- `feedback_state`: System status and element state - selected, disabled, loading, errors, empty states.
- `consistency`: The same kind of element is styled or worded differently within the screen.
- `missing_function`: A control, option or piece of information the task needs is absent.
- `aesthetics`: Overall visual appeal without a specific usability mechanism.
- `other`: None of the above; say what in the note.

Switch the model to `"gpt5"` or `"gemini3_1pro"` when a model is done.

In [ ]:
review("matched", "claude4_7")

In [ ]:
# paste the confirm(...) line printed above

## 4.2 Unmatched expert comments

*Why did the model miss it?*

**cause** (take the first that applies)
- `judge_miss`: A comment on the same task covers it; the judge should have matched them.
- `other_task_match`: A comment the model wrote for another task on the same screen covers it (pairs are only judged within a task).
- `expert_incorrect`: The expert's claim is contradicted by the screenshot.
- `unverifiable`: Cannot be checked against the screenshot - too vague or taste-based ("unappealing", "lacks balance"), or depends on interaction or flow the screenshot does not show.
- `same_problem_other_element`: The model raised the same kind of problem, but on a different element.
- `clear_omission`: Specific and visible on the screenshot, and no comment on the screen covers it.

**issue_type**
- `contrast_legibility`: Text or icons hard to read - colour contrast, small or light text.
- `hierarchy_emphasis`: Wrong visual weight - an important element is not prominent, or an unimportant one dominates.
- `layout_spacing`: Alignment, margins, whitespace, balance, clipped or cut-off content.
- `affordance`: Unclear whether or how an element is interactive, or what kind of control it is.
- `touch_target`: Controls too small or too close together to tap reliably.
- `navigation`: Missing or unclear way to move between screens (back, close, menu structure).
- `labeling_content`: Wording, labels, terminology, icon meaning, copy errors.
- `feedback_state`: System status and element state - selected, disabled, loading, errors, empty states.
- `consistency`: The same kind of element is styled or worded differently within the screen.
- `missing_function`: A control, option or piece of information the task needs is absent.
- `aesthetics`: Overall visual appeal without a specific usability mechanism.
- `other`: None of the above; say what in the note.

Switch the model to `"gpt5"` or `"gemini3_1pro"` when a model is done.

In [ ]:
review("unmatched", "claude4_7")

In [ ]:
# paste the confirm(...) line printed above

## 4.3 Unsupported LLM comments (blind: judge against the screenshot only)

*Is the unsupported comment a legitimate observation the expert did not make?*

**validity** (take the first that applies)
- `hallucination`: The core claim is contradicted by the screenshot - the element, text, colour or state is not as described (exaggerated wording about something real is not a hallucination).
- `unverifiable`: Cannot be checked against the screenshot - depends on behaviour, state or content it does not show, or the image is too unclear.
- `valid_generic`: Accurate, but a checklist best practice that would fit most screens (required-field markers, filter/sort, labels on icons).
- `valid_specific`: Accurate and specific to this screen.

**issue_type**
- `contrast_legibility`: Text or icons hard to read - colour contrast, small or light text.
- `hierarchy_emphasis`: Wrong visual weight - an important element is not prominent, or an unimportant one dominates.
- `layout_spacing`: Alignment, margins, whitespace, balance, clipped or cut-off content.
- `affordance`: Unclear whether or how an element is interactive, or what kind of control it is.
- `touch_target`: Controls too small or too close together to tap reliably.
- `navigation`: Missing or unclear way to move between screens (back, close, menu structure).
- `labeling_content`: Wording, labels, terminology, icon meaning, copy errors.
- `feedback_state`: System status and element state - selected, disabled, loading, errors, empty states.
- `consistency`: The same kind of element is styled or worded differently within the screen.
- `missing_function`: A control, option or piece of information the task needs is absent.
- `aesthetics`: Overall visual appeal without a specific usability mechanism.
- `other`: None of the above; say what in the note.

Switch the model to `"gpt5"` or `"gemini3_1pro"` when a model is done.

In [ ]:
review("unsupported", "claude4_7")

In [ ]:
# paste the confirm(...) line printed above

## 4.4 Contradictions (rows without codes are new since the judge re-run)

*Do the two comments really contradict each other?*

**label**
- `0`: Not a contradiction.
- `1`: Partial contradiction.
- `2`: True contradiction.

Switch the model to `"gpt5"` or `"gemini3_1pro"` when a model is done.

In [ ]:
review("contradiction", "claude4_7")

In [ ]:
# paste the confirm(...) line printed above

## Finish

When `status()` shows `remaining` = 0 everywhere, run `2_b_comment_matching_analysis.ipynb` top to bottom (4.5 rebuilds
the summary tables from the reviewed codes), then delete this notebook and `_temp_audit_review_progress.csv`.

In [ ]:
display(status())
for section in SECTIONS:
    left = is_unsure(load(section)).sum()
    if left:
        print(f"{section}: {left} notes still start with [unsure] or [recheck]")